# **|| Safety Compliance Detection System ||**

A real-time PPE (Personal Protective Equipment) detection system trained on construction site safety data.

**Detects:**
- Helmet / Hardhat
- Safety Vest
- Face Mask
- No-Helmet / No-Vest / No-Mask (non-compliance)

**Input modes:**
- Upload an image
- Upload a video file
- Webcam capture

**Tech used:** `YOLOv8` · `OpenCV` · `ultralytics` · `Roboflow`


## **1** - ***Installation***

In [ ]:
%%capture
!pip install ultralytics roboflow opencv-python-headless Pillow

## **2** - ***Imports and Settings***

In [ ]:
import os
import sys
import cv2
import time
import numpy as np
from PIL import Image
from IPython.display import display, HTML
from base64 import b64encode, b64decode
import matplotlib.pyplot as plt
from ultralytics import YOLO

os.environ["PYTHONIOENCODING"] = "utf-8"

# --- Settings ---
CONFIDENCE_THRESHOLD = 0.4   # Minimum confidence to show a detection
EPOCHS = 20                  # Number of training epochs — increase for better accuracy but takes longer
IMAGE_SIZE = 640             # Input image size for training
BASE_MODEL = "yolov8n.pt"    # Starting model for training - yolov8s.pt gives better results but takes more time

# Colors for drawing boxes — BGR format for OpenCV
COLOR_OK    = (0, 255, 0)    # green — PPE detected
COLOR_ALERT = (0, 0, 255)    # red — PPE missing
COLOR_INFO  = (255, 165, 0)  # orange — general detection

print("Imports done!")

## **3** - ***Download PPE Dataset***

We use the **Construction Site Safety** dataset from Roboflow. It contains images of workers with labels for helmet, vest, mask and their missing counterparts.

You need a free Roboflow account to get the API key. Sign up at https://roboflow.com

In [ ]:
from getpass import getpass
from roboflow import Roboflow

# Free API key from https://app.roboflow.com — go to Settings > API Keys
ROBOFLOW_API_KEY = getpass("Enter your Roboflow API key: ")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Construction Site Safety dataset — has hardhat, mask, vest, no-hardhat, no-mask, no-vest, person classes
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
dataset = project.version(1).download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")
DATA_YAML = os.path.join(dataset.location, "data.yaml")
print(f"Data config: {DATA_YAML}")

## **4** - ***Check the Dataset***

Let's quickly see how many images we have and what classes are in the dataset before training.

In [ ]:
import yaml

with open(DATA_YAML, "r") as f:
    data_info = yaml.safe_load(f)

print("Dataset info:")
print(f"  Classes: {data_info['names']}")
print(f"  Number of classes: {data_info['nc']}")

# Count images in train/val splits
for split in ["train", "val", "test"]:
    split_path = os.path.join(dataset.location, split, "images")
    if os.path.exists(split_path):
        count = len(os.listdir(split_path))
        print(f"  {split} images: {count}")

# Show a few sample images from the dataset
train_img_path = os.path.join(dataset.location, "train", "images")
sample_images = os.listdir(train_img_path)[:4]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, img_name in enumerate(sample_images):
    img = cv2.imread(os.path.join(train_img_path, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(f"Sample {i+1}")

plt.suptitle("Sample Training Images")
plt.tight_layout()
plt.show()

## **5** - ***Train the Model***

We fine-tune YOLOv8 on the PPE dataset. Make sure you have **GPU enabled** in Colab. Go to Runtime > Change runtime type > T4 GPU.

Training for 20 epochs takes around 15-25 minutes on a T4 GPU.

In [ ]:
import torch

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")
if device == "cpu":
    print("Warning: Training on CPU will be very slow. Enable GPU in Runtime > Change runtime type")

# Load the base YOLOv8 model
model = YOLO(BASE_MODEL)

print(f"Starting training for {EPOCHS} epochs...")
print("This will take around 15-25 minutes on GPU")

# Train the model on our PPE dataset
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=16,
    device=device,
    project="ppe_training",
    name="safety_model",
    exist_ok=True,
    verbose=True
)

print("Training complete!")

## **6** - ***Evaluate the Model***

Check how well the model performs on the validation set.

In [ ]:
# Path to the best trained model weights
TRAINED_MODEL_PATH = "/content/runs/detect/ppe_training/safety_model/weights/best.pt"

# Load the trained model
trained_model = YOLO(TRAINED_MODEL_PATH)

# Run evaluation on the validation set
print("Evaluating on validation set...")
metrics = trained_model.val(data=DATA_YAML)

print(f"\nmAP50: {metrics.box.map50:.3f}")       # Mean average precision at IoU=0.5
print(f"mAP50-95: {metrics.box.map:.3f}")        # Mean average precision across IoU thresholds
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

# Show training curves
results_img = cv2.imread("ppe_training/safety_model/results.png")
if results_img is not None:
    results_img = cv2.cvtColor(results_img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 6))
    plt.imshow(results_img)
    plt.axis("off")
    plt.title("Training Results")
    plt.show()
else:
    print("Training curves image not found")

## **7** - ***Helper Functions***

In [ ]:
# Use the trained model from here on
model = trained_model

# These are the classes from the construction site safety dataset
COMPLIANT_CLASSES = ["hardhat", "mask", "safety vest", "vest"]
VIOLATION_CLASSES = ["no-hardhat", "no-mask", "no-safety vest", "no-vest"]


def run_detection(frame):
    """Runs the trained model on a single frame"""
    results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
    return results[0]


def check_compliance(detected_labels):
    """
    Checks which PPE items are present and which are missing.
    Returns a compliance dict and a list of violations.
    """
    detected_lower = [l.lower() for l in detected_labels]

    compliance = {
        "helmet": False,
        "vest":   False,
        "mask":   False
    }

    violations = []

    for label in detected_lower:
        if "hardhat" in label or "helmet" in label:
            if "no" not in label:
                compliance["helmet"] = True
            else:
                violations.append("No Helmet")

        if "vest" in label:
            if "no" not in label:
                compliance["vest"] = True
            else:
                violations.append("No Vest")

        if "mask" in label:
            if "no" not in label:
                compliance["mask"] = True
            else:
                violations.append("No Mask")

    return compliance, violations


def generate_alert(compliance, violations):
    """Returns an alert message and color based on compliance status"""
    missing = [item for item, present in compliance.items() if not present]

    if violations:
        msg = "ALERT - Violations: " + ", ".join(violations)
        return msg, COLOR_ALERT
    elif not any(compliance.values()):
        return "WARNING - No PPE detected", COLOR_ALERT
    else:
        return "COMPLIANT - PPE detected", COLOR_OK


def draw_results(frame, result, alert_message, alert_color):
    """Draws bounding boxes and alert banner on the frame"""
    annotated = frame.copy()

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        class_id = int(box.cls[0])
        label = model.names[class_id]

        # red box for violations, green for compliant gear
        if "no-" in label.lower():
            box_color = COLOR_ALERT
        else:
            box_color = COLOR_OK

        cv2.rectangle(annotated, (x1, y1), (x2, y2), box_color, 2)
        cv2.putText(annotated, f"{label} {conf:.2f}", (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, box_color, 2)

    # Alert banner at the top
    cv2.rectangle(annotated, (0, 0), (annotated.shape[1], 40), alert_color, -1)
    cv2.putText(annotated, alert_message, (10, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)

    return annotated


print("Helper functions ready!")

## **8** - ***Image Detection***

Upload an image to test the trained model.

In [ ]:
from google.colab import files

print("Upload an image (jpg or png):")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]

    img = cv2.imread(filename)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    print("Running detection...")
    result = run_detection(img_rgb)

    detected_labels = [model.names[int(c)] for c in result.boxes.cls]
    print(f"Detected: {detected_labels}")

    compliance, violations = check_compliance(detected_labels)
    alert_message, alert_color = generate_alert(compliance, violations)

    print(f"\nCompliance Status:")
    for item, present in compliance.items():
        print(f"  {item}: {'Found' if present else 'NOT Found'}")
    print(f"\n{alert_message}")

    annotated = draw_results(img_rgb, result, alert_message, alert_color)

    plt.figure(figsize=(12, 8))
    plt.imshow(annotated)
    plt.axis("off")
    plt.title(alert_message)
    plt.tight_layout()
    plt.savefig("detection_result.jpg", dpi=150, bbox_inches="tight")
    plt.show()
    print("Result saved as detection_result.jpg")
else:
    print("No file uploaded.")

## **9** - ***Video Detection***

Upload a video file and run detection frame by frame.

In [ ]:
from google.colab import files
import io

print("Upload a video file (mp4 or avi):")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]

    cap = cv2.VideoCapture(filename)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Video info: {total_frames} frames, {fps} FPS, {width}x{height}")

    output_path = "output_detection.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    alert_counts = {"COMPLIANT": 0, "ALERT": 0}
    SKIP_FRAMES = 2  # Process every 2nd frame to speed things up

    print("Processing video...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        if frame_count % SKIP_FRAMES != 0:
            out.write(frame)
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = run_detection(frame_rgb)

        detected_labels = [model.names[int(c)] for c in result.boxes.cls]
        compliance, violations = check_compliance(detected_labels)
        alert_message, alert_color = generate_alert(compliance, violations)

        if "COMPLIANT" in alert_message:
            alert_counts["COMPLIANT"] += 1
        else:
            alert_counts["ALERT"] += 1

        annotated_rgb = draw_results(frame_rgb, result, alert_message, alert_color)
        annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)
        out.write(annotated_bgr)

        if frame_count % 30 == 0:
            print(f"  Processed {frame_count}/{total_frames} frames...")

    cap.release()
    out.release()

    print(f"\nDone! Processed {frame_count} frames")
    print(f"Compliant frames: {alert_counts['COMPLIANT']}")
    print(f"Alert frames: {alert_counts['ALERT']}")

    # Play output video inline
    mp4 = open(output_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>'))
else:
    print("No file uploaded.")

## **10** - ***Webcam Detection***

Captures a frame from your webcam and runs detection on it. Allow camera access when the browser asks.

In [ ]:
from IPython.display import Javascript
from google.colab.output import eval_js
import PIL
import io

def capture_webcam_frame():
    """Opens the webcam in the browser and captures one frame"""
    js = Javascript("""
        async function takePhoto() {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capture Frame';
            div.appendChild(capture);
            document.body.appendChild(div);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(video);
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            video.remove();

            return canvas.toDataURL('image/jpeg', 0.8);
        }
        takePhoto();
    """)
    display(js)
    data = eval_js('takePhoto()')
    binary = b64decode(data.split(',')[1])
    img = PIL.Image.open(io.BytesIO(binary))
    return np.array(img)


print("Click 'Capture Frame' to take a photo from your webcam...")

try:
    frame = capture_webcam_frame()

    print("Running detection...")
    result = run_detection(frame)

    detected_labels = [model.names[int(c)] for c in result.boxes.cls]
    compliance, violations = check_compliance(detected_labels)
    alert_message, alert_color = generate_alert(compliance, violations)

    print(f"Detected: {detected_labels}")
    print(f"\nCompliance Status:")
    for item, present in compliance.items():
        print(f"  {item}: {'Found' if present else 'NOT Found'}")
    print(f"\n{alert_message}")

    annotated = draw_results(frame, result, alert_message, alert_color)

    plt.figure(figsize=(10, 7))
    plt.imshow(annotated)
    plt.axis("off")
    plt.title(alert_message)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Webcam error: {e}")
    print("Make sure you allow camera access when the browser asks.")

## **11** - ***Save Trained Model***

Downloads the trained model weights so you don't have to retrain next time.

In [ ]:
from google.colab import files

# Download the best weights file
if os.path.exists(TRAINED_MODEL_PATH):
    files.download(TRAINED_MODEL_PATH)
    print("Model downloaded as best.pt")
    print("Next time you can load it directly with: model = YOLO('best.pt')")
else:
    print("Trained model not found — make sure training completed successfully")

## ****Things You Can Change!***

- `EPOCHS` — increase to 50 for better accuracy, decrease to 10 for a quick test run
- `CONFIDENCE_THRESHOLD` — lower to 0.3 to catch more detections, raise to 0.6 to reduce false positives
- `BASE_MODEL` — swap to `yolov8s.pt` or `yolov8m.pt` for a stronger starting point (slower training)
- `SKIP_FRAMES` in the video cell — set to 1 to process every single frame
- `batch=16` in training — lower to 8 if you run into memory errors on GPU
